# 01 — CO₂ Data Generation

Generate CoolProp CO₂ data, define the first ΔZ-based regimes, and create temperature-separated natural and stress splits.

In [ ]:
!pip install CoolProp pandas numpy matplotlib seaborn

In [ ]:
from pathlib import Path


def find_project_root():
    current = Path.cwd()
    candidates = [current, current.parent]

    for candidate in candidates:
        if (candidate / "notebooks").exists():
            return candidate
        if (candidate / "data").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", DATA_DIR)


In [ ]:
import CoolProp.CoolProp as CP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import TwoSlopeNorm

FLUID = "CO2"
R = 8.31446261815324  # J / (mol K)

T_MIN = 310.0
T_MAX = 500.0
NUM_T = 50

RHO_MIN = 100.0
RHO_MAX = 20000.0
NUM_RHO = 100

T_CRITICAL = CP.PropsSI("Tcrit", FLUID)
RHO_CRITICAL = CP.PropsSI("rhomolar_critical", FLUID)

print(f"Fluid: {FLUID}")
print(f"Critical temperature: {T_CRITICAL:.6f} K")
print(f"Critical molar density: {RHO_CRITICAL:.6f} mol/m³")

In [ ]:
temperature_values = np.linspace(
    T_MIN,
    T_MAX,
    NUM_T
)

density_values = np.geomspace(
    RHO_MIN,
    RHO_MAX,
    NUM_RHO
)

print(f"Number of temperatures: {len(temperature_values)}")
print(f"Number of densities: {len(density_values)}")
print(f"Expected data points: {NUM_T * NUM_RHO}")

In [ ]:
records = []
failed_points = []

for temperature in temperature_values:
    for molar_density in density_values:
        try:
            real_pressure = CP.PropsSI(
                "P",
                "T", temperature,
                "Dmolar", molar_density,
                FLUID
            )

            phase = CP.PhaseSI(
                "T", temperature,
                "Dmolar", molar_density,
                FLUID
            )

            ideal_pressure = molar_density * R * temperature

            compressibility_factor = (
                real_pressure / ideal_pressure
            )

            delta_z = compressibility_factor - 1.0

            reduced_temperature = (
                temperature / T_CRITICAL
            )

            reduced_density = (
                molar_density / RHO_CRITICAL
            )

            values = [
                real_pressure,
                ideal_pressure,
                compressibility_factor,
                delta_z
            ]

            if not np.all(np.isfinite(values)):
                raise ValueError("Non-finite value detected")

            records.append({
                "T_K": temperature,
                "rho_mol_m3": molar_density,
                "p_real_Pa": real_pressure,
                "p_ideal_Pa": ideal_pressure,
                "T_reduced": reduced_temperature,
                "rho_reduced": reduced_density,
                "Z": compressibility_factor,
                "delta_Z": delta_z,
                "phase": phase
            })

        except Exception as error:
            failed_points.append({
                "T_K": temperature,
                "rho_mol_m3": molar_density,
                "error": str(error)
            })

df = pd.DataFrame(records)
failed_df = pd.DataFrame(failed_points)

print(f"Valid points: {len(df)}")
print(f"Failed points: {len(failed_df)}")

df.head()

In [ ]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated(
    subset=["T_K", "rho_mol_m3"]
).sum())

print("\nDelta Z statistics:")
print(df["delta_Z"].describe())

print("\nPhase counts:")
print(df["phase"].value_counts())

print("\nSign counts:")
print({
    "delta_Z_negative": int((df["delta_Z"] < 0).sum()),
    "delta_Z_near_zero": int(
        np.isclose(df["delta_Z"], 0.0, atol=1e-6).sum()
    ),
    "delta_Z_positive": int((df["delta_Z"] > 0).sum())
})

In [ ]:
low_density_threshold = df["rho_mol_m3"].quantile(0.05)

low_density_df = df[
    df["rho_mol_m3"] <= low_density_threshold
]

print(f"Low-density threshold: {low_density_threshold:.3f} mol/m³")
print(f"Mean Z at low density: {low_density_df['Z'].mean():.8f}")
print(
    "Mean absolute delta Z at low density:",
    f"{low_density_df['delta_Z'].abs().mean():.8f}"
)
print(
    "Maximum absolute delta Z at low density:",
    f"{low_density_df['delta_Z'].abs().max():.8f}"
)

In [ ]:
EPSILON = 0.03

def determine_regime(delta_z):
    if abs(delta_z) <= EPSILON:
        return "near_ideal"

    if delta_z < -EPSILON:
        return "attraction_dominated"

    return "excluded_volume_dominated"

df["regime"] = df["delta_Z"].apply(determine_regime)

regime_counts = df["regime"].value_counts()
regime_percentages = (
    df["regime"].value_counts(normalize=True) * 100
)

regime_summary = pd.DataFrame({
    "count": regime_counts,
    "percentage": regime_percentages
})

regime_summary

In [ ]:
heatmap_data = df.pivot(
    index="rho_reduced",
    columns="T_reduced",
    values="delta_Z"
).sort_index().sort_index(axis=1)

x_values = heatmap_data.columns.to_numpy()
y_values = heatmap_data.index.to_numpy()
z_values = heatmap_data.to_numpy()

normalization = TwoSlopeNorm(
    vmin=np.nanmin(z_values),
    vcenter=0.0,
    vmax=np.nanmax(z_values)
)

plt.figure(figsize=(10, 7))

heatmap = plt.pcolormesh(
    x_values,
    y_values,
    z_values,
    shading="auto",
    cmap="coolwarm",
    norm=normalization
)

plt.contour(
    x_values,
    y_values,
    z_values,
    levels=[-EPSILON, 0.0, EPSILON],
    colors=["black", "white", "black"],
    linewidths=[1.0, 1.5, 1.0]
)

plt.yscale("log")

plt.xlabel("Reduced temperature $T_r$")
plt.ylabel("Reduced molar density $\\rho_r$")
plt.title("Compressibility Residual $\\Delta Z$ for CO₂")

colorbar = plt.colorbar(heatmap)
colorbar.set_label("$\\Delta Z = Z - 1$")

plt.tight_layout()
plt.show()

In [ ]:
output_filename = DATA_DIR / "co2_dataset.csv"

df.to_csv(
    output_filename,
    index=False
)

print(f"Saved {len(df)} rows to {output_filename}")


In [ ]:
epsilon_values = [0.01, 0.03, 0.05, 0.10]

epsilon_results = []

for epsilon in epsilon_values:
    near_ideal = np.abs(df["delta_Z"]) <= epsilon
    attraction = df["delta_Z"] < -epsilon
    excluded_volume = df["delta_Z"] > epsilon

    epsilon_results.append({
        "epsilon": epsilon,
        "near_ideal_count": near_ideal.sum(),
        "near_ideal_percentage": near_ideal.mean() * 100,
        "attraction_count": attraction.sum(),
        "attraction_percentage": attraction.mean() * 100,
        "excluded_volume_count": excluded_volume.sum(),
        "excluded_volume_percentage": excluded_volume.mean() * 100
    })

epsilon_summary = pd.DataFrame(epsilon_results)

epsilon_summary

In [ ]:
excluded_df = df[df["delta_Z"] > 0.03].copy()

excluded_df["p_real_MPa"] = (
    excluded_df["p_real_Pa"] / 1e6
)

excluded_df[
    [
        "T_K",
        "rho_mol_m3",
        "p_real_MPa",
        "T_reduced",
        "rho_reduced",
        "delta_Z"
    ]
].describe()

In [ ]:
regime_ranges = df.groupby("regime").agg(
    count=("delta_Z", "size"),
    T_min=("T_K", "min"),
    T_max=("T_K", "max"),
    rho_min=("rho_mol_m3", "min"),
    rho_max=("rho_mol_m3", "max"),
    delta_Z_min=("delta_Z", "min"),
    delta_Z_max=("delta_Z", "max")
)

regime_ranges

In [ ]:
dense_temperatures = np.linspace(
    360.0,
    500.0,
    80
)

dense_molar_densities = np.linspace(
    12000.0,
    20000.0,
    80
)

dense_records = []
dense_failed_points = []

for temperature in dense_temperatures:
    for molar_density in dense_molar_densities:
        try:
            real_pressure = CP.PropsSI(
                "P",
                "T", temperature,
                "Dmolar", molar_density,
                FLUID
            )

            phase = CP.PhaseSI(
                "T", temperature,
                "Dmolar", molar_density,
                FLUID
            )

            ideal_pressure = molar_density * R * temperature
            compressibility_factor = real_pressure / ideal_pressure
            delta_z = compressibility_factor - 1.0

            values = [
                real_pressure,
                ideal_pressure,
                compressibility_factor,
                delta_z
            ]

            if not np.all(np.isfinite(values)):
                raise ValueError("Non-finite value detected")

            dense_records.append({
                "T_K": temperature,
                "rho_mol_m3": molar_density,
                "p_real_Pa": real_pressure,
                "p_ideal_Pa": ideal_pressure,
                "T_reduced": temperature / T_CRITICAL,
                "rho_reduced": molar_density / RHO_CRITICAL,
                "Z": compressibility_factor,
                "delta_Z": delta_z,
                "phase": phase
            })

        except Exception as error:
            dense_failed_points.append({
                "T_K": temperature,
                "rho_mol_m3": molar_density,
                "error": str(error)
            })

dense_df = pd.DataFrame(dense_records)

print("Valid dense points:", len(dense_df))
print("Failed dense points:", len(dense_failed_points))

In [ ]:
EPSILON = 0.03

dense_df["regime"] = dense_df["delta_Z"].apply(
    determine_regime
)

dense_regime_summary = pd.DataFrame({
    "count": dense_df["regime"].value_counts(),
    "percentage":
        dense_df["regime"].value_counts(normalize=True) * 100
})

dense_regime_summary

In [ ]:
training_df = pd.concat(
    [df, dense_df],
    ignore_index=True
)

training_df = training_df.drop_duplicates(
    subset=["T_K", "rho_mol_m3"]
).reset_index(drop=True)

training_df["regime"] = training_df["delta_Z"].apply(
    determine_regime
)

training_summary = pd.DataFrame({
    "count": training_df["regime"].value_counts(),
    "percentage":
        training_df["regime"].value_counts(normalize=True) * 100
})

print("Total training candidate points:", len(training_df))

training_summary

In [ ]:
# 原始固定網格：未來作為公平 evaluation dataset
df.to_csv(
    DATA_DIR / "co2_evaluation_grid.csv",
    index=False
)

# 含高密度加密資料：未來用於 training
training_df.to_csv(
    DATA_DIR / "co2_training_candidate.csv",
    index=False
)

print("Saved processed candidate datasets to:", DATA_DIR)


In [ ]:
original_temperatures = np.sort(
    df["T_K"].unique()
)

test_temperatures = original_temperatures[::5]
validation_temperatures = original_temperatures[2::5]

print("Test temperatures:")
print(test_temperatures)

print("\nValidation temperatures:")
print(validation_temperatures)

print("\nNumber of test isotherms:", len(test_temperatures))
print(
    "Number of validation isotherms:",
    len(validation_temperatures)
)

In [ ]:
def matches_any_temperature(series, temperatures):
    result = np.zeros(len(series), dtype=bool)

    for temperature in temperatures:
        result |= np.isclose(
            series.to_numpy(),
            temperature,
            atol=1e-9
        )

    return result


test_mask_original = matches_any_temperature(
    df["T_K"],
    test_temperatures
)

validation_mask_original = matches_any_temperature(
    df["T_K"],
    validation_temperatures
)

test_df = df[test_mask_original].copy()
validation_df = df[validation_mask_original].copy()


held_out_temperatures = np.concatenate([
    test_temperatures,
    validation_temperatures
])

held_out_mask_training = matches_any_temperature(
    training_df["T_K"],
    held_out_temperatures
)

train_df = training_df[
    ~held_out_mask_training
].copy()


train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training points:", len(train_df))
print("Validation points:", len(validation_df))
print("Test points:", len(test_df))

In [ ]:
train_temperatures = set(
    np.round(train_df["T_K"], 10)
)

validation_temperature_set = set(
    np.round(validation_df["T_K"], 10)
)

test_temperature_set = set(
    np.round(test_df["T_K"], 10)
)

print(
    "Train-validation overlap:",
    train_temperatures & validation_temperature_set
)

print(
    "Train-test overlap:",
    train_temperatures & test_temperature_set
)

print(
    "Validation-test overlap:",
    validation_temperature_set & test_temperature_set
)

In [ ]:
def summarize_split(dataset, split_name):
    counts = dataset["regime"].value_counts()
    percentages = (
        dataset["regime"].value_counts(normalize=True) * 100
    )

    summary = pd.DataFrame({
        "split": split_name,
        "regime": counts.index,
        "count": counts.values,
        "percentage": percentages.values
    })

    return summary


split_summary = pd.concat([
    summarize_split(train_df, "train"),
    summarize_split(validation_df, "validation"),
    summarize_split(test_df, "test")
], ignore_index=True)

split_summary

In [ ]:
split_ranges = pd.DataFrame({
    "split": ["train", "validation", "test"],

    "rows": [
        len(train_df),
        len(validation_df),
        len(test_df)
    ],

    "T_min": [
        train_df["T_K"].min(),
        validation_df["T_K"].min(),
        test_df["T_K"].min()
    ],

    "T_max": [
        train_df["T_K"].max(),
        validation_df["T_K"].max(),
        test_df["T_K"].max()
    ],

    "rho_min": [
        train_df["rho_mol_m3"].min(),
        validation_df["rho_mol_m3"].min(),
        test_df["rho_mol_m3"].min()
    ],

    "rho_max": [
        train_df["rho_mol_m3"].max(),
        validation_df["rho_mol_m3"].max(),
        test_df["rho_mol_m3"].max()
    ]
})

split_ranges

In [ ]:
train_df.to_csv(
    DATA_DIR / "co2_train.csv",
    index=False
)

validation_df.to_csv(
    DATA_DIR / "co2_validation.csv",
    index=False
)

test_df.to_csv(
    DATA_DIR / "co2_test.csv",
    index=False
)

print("Saved natural train/validation/test datasets to:", DATA_DIR)


In [ ]:
dense_unique_temperatures = np.sort(
    dense_df["T_K"].unique()
)

# 排除已被 natural validation/test 使用的溫度
available_dense_temperatures = []

for temperature in dense_unique_temperatures:
    matches_held_out = np.any(
        np.isclose(
            held_out_temperatures,
            temperature,
            atol=1e-9
        )
    )

    if not matches_held_out:
        available_dense_temperatures.append(temperature)

available_dense_temperatures = np.array(
    available_dense_temperatures
)

stress_test_temperatures = (
    available_dense_temperatures[1::8]
)

stress_validation_temperatures = (
    available_dense_temperatures[4::8]
)

print(
    "Stress validation isotherms:",
    len(stress_validation_temperatures)
)

print(
    "Stress test isotherms:",
    len(stress_test_temperatures)
)

In [ ]:
stress_validation_mask = matches_any_temperature(
    dense_df["T_K"],
    stress_validation_temperatures
)

stress_test_mask = matches_any_temperature(
    dense_df["T_K"],
    stress_test_temperatures
)

stress_validation_df = dense_df[
    stress_validation_mask
].copy().reset_index(drop=True)

stress_test_df = dense_df[
    stress_test_mask
].copy().reset_index(drop=True)

stress_validation_df["regime"] = (
    stress_validation_df["delta_Z"].apply(
        determine_regime
    )
)

stress_test_df["regime"] = (
    stress_test_df["delta_Z"].apply(
        determine_regime
    )
)

print(
    "Stress validation points:",
    len(stress_validation_df)
)

print(
    "Stress test points:",
    len(stress_test_df)
)

In [ ]:
all_stress_temperatures = np.concatenate([
    stress_validation_temperatures,
    stress_test_temperatures
])

stress_overlap_mask = matches_any_temperature(
    train_df["T_K"],
    all_stress_temperatures
)

train_df = train_df[
    ~stress_overlap_mask
].copy().reset_index(drop=True)

print(
    "Updated training points:",
    len(train_df)
)

In [ ]:
stress_split_summary = pd.concat([
    summarize_split(
        stress_validation_df,
        "stress_validation"
    ),
    summarize_split(
        stress_test_df,
        "stress_test"
    )
], ignore_index=True)

stress_split_summary

In [ ]:
train_temperature_values = set(
    np.round(train_df["T_K"], 10)
)

stress_validation_temperature_set = set(
    np.round(
        stress_validation_df["T_K"],
        10
    )
)

stress_test_temperature_set = set(
    np.round(
        stress_test_df["T_K"],
        10
    )
)

print(
    "Train-stress-validation overlap:",
    train_temperature_values
    & stress_validation_temperature_set
)

print(
    "Train-stress-test overlap:",
    train_temperature_values
    & stress_test_temperature_set
)

print(
    "Stress-validation-test overlap:",
    stress_validation_temperature_set
    & stress_test_temperature_set
)

In [ ]:
train_df.to_csv(
    DATA_DIR / "co2_train.csv",
    index=False
)

stress_validation_df.to_csv(
    DATA_DIR / "co2_stress_validation.csv",
    index=False
)

stress_test_df.to_csv(
    DATA_DIR / "co2_stress_test.csv",
    index=False
)

print("Saved final datasets to:", DATA_DIR)
